# Variants specific to one group

One objective: show the transcript isoforms, ORFs and splicing variants that are found in one group and not in the other. Run it on SpA against not SpA, and again on B27-positive against B27-negative SpA by changing `paths.json`.

In [ ]:
source("src/heatmap.R")
source("src/specific.R")
source("src/paths.R")
options(repr.plot.width = 8, repr.plot.height = 4)
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
min_count <- c(isoforms = 1, orfs = 1, variants = 5)   # reads needed in a sample to call a feature detected
k <- 2                                                  # samples of a group that must have it
exclude <- c("420_20190402")                            # samples to leave out in the second drawing
genes <- c("HLA-B", "ERAP1", "ERAP2")                   # genes to list
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
exclude <- intersect(exclude, groups$sample)
table(groups$group)

**Explanation.** A feature is called *detected* in a sample when it has at least `min_count` reads there, and *specific* to a group when it is detected in at least `k` samples of that group and in none of the other group. `k` is never larger than the size of the smaller group. The counts are the pipeline's own; `variants` are the splicing variants (introns) of the long-read leafcutter output. The table shows how many samples are in each group, which limits what `k` can be. `exclude` names samples to leave out in the second drawing. A larger group has more chances to detect a feature, so when the groups differ in size (19 SpA against 5 not SpA, for example) more features come out specific to the larger group by chance alone. Compare the two groups' counts with that in mind, and rely on the heatmaps and the per-gene lists more than on the raw totals.

In [ ]:
feat <- list(
  isoforms = read_features("isoforms", file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_raw_CPM_matrix.txt")), groups),
  orfs     = read_features("orfs",     file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_raw_CPM_matrix.txt")), groups),
  variants = read_features("variants", file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.subisoform_clusters.txt")), groups))
tabs <- Map(function(f, m) specificity(f, groups, min_count = m, k = k), feat, min_count[names(feat)])
specific_summary(tabs)

**Explanation.** For every transcript isoform, ORF and splicing variant, this counts in how many samples of each group it is detected and decides whether it is specific to one group. The table below the code counts the features that are specific to each group and those that are not.

In [ ]:
save_plot(specific_barplot(specific_summary(tabs), title = paste0("Features specific to a group (detected in at least ", attr(tabs$isoforms, "k"), " samples, none in the other group)")), "specific_counts")

**Explanation.** One bar per group and level: how many features are found only in that group. The grey bar is everything else, including features seen in both groups and features too rare to call. A feature seen in no samples of the other group may only be unsampled, because long-read depth is shallow, so treat these as leads to check, not as proof of absence.

In [ ]:
specific_genes(tabs, genes)

**Explanation.** The specific features of the genes in `genes`, with the number of samples of each group (`n_...`) and the average count (`mean_...`). If a gene is missing here, none of its features was specific. HLA-B is highly variable between people and B27 is the SpA allele, so a feature specific to B27-positive samples may reflect the allele and read mapping and not splicing.

In [ ]:
lapply(tabs, function(t) head(specific_by_gene(t), 10))

**Explanation.** The ten genes with the most specific features at each level, and how many are specific to each group. A gene with several specific variants for the same group is a better lead than a single one.

In [ ]:
for (l in names(tabs)) show_heatmap(specific_heatmap(feat[[l]], tabs[[l]], groups, title = paste0("Specific ", l, ", all samples")), paste0("specific_", l, "_heatmap"))

**Explanation.** One heatmap per level of only the specific features. Rows are grouped by the group they belong to; columns are samples, split by group. The colour is log2(count + 1). A convincing feature is present in most samples of its group, not in one sample only. Every row has its name.

In [ ]:
dir.create("tables", showWarnings = FALSE)
for (l in names(tabs)) write.table(tabs[[l]], paste0(figure_stem(paste0("specific_", l), "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
list.files("tables", pattern = comparison)

**Explanation.** Each table is saved in `tables/` with the comparison name, one file per level. `igv_files.ipynb` reads them to write coloured IGV tracks of the specific features.

In [ ]:
if (length(exclude) > 0) {
  g2 <- drop_samples(groups, exclude)
  label <- paste(exclude, collapse = ", ")
  tabs2 <- Map(function(f, m) specificity(use_samples(f, g2), g2, min_count = m, k = k), feat, min_count[names(feat)])
  print(specific_summary(tabs2))
  save_plot(specific_barplot(specific_summary(tabs2), title = paste0("Features specific to a group, without ", label)), "specific_counts_without")
  for (l in names(tabs2)) {
    show_heatmap(specific_heatmap(feat[[l]], tabs2[[l]], g2, title = paste0("Specific ", l, ", without ", label)), paste0("specific_", l, "_heatmap_without"))
    write.table(tabs2[[l]], paste0(figure_stem(paste0("specific_", l, "_without"), "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
  }
} else {
  message("no sample to leave out: no second drawing")
}

**Explanation.** The same without the samples in `exclude`, when any of them are in this run. Features that stay specific are not driven by one sample. With a sample removed, a group has fewer samples, so fewer features can reach `k`.